# 01. LOTA Forensic Pipeline: Mathematical & Visual Validation
**Paper**: *LOTA: Bit-Planes Guided AI-Generated Image Detection* (ICCV 2025)

This notebook supports two distinct validation modes:
- **Mode A (Synthetic Mathematical Validation)**: Validates mathematical correctness (bit slicing, normalization, and MGPS gradient convolutions) using controlled synthetic samples.
- **Mode B (Real GenImage Forensic Validation)**: Performs genuine forensic inspection across Real (nature) and AI-Generated (ai) image pairs from GenImage.

> **Integrity Rule**: If the GenImage dataset is absent, the notebook runs strictly in Mode A and produces an explicit warning. Qualitative comparisons with Figure 3 of the paper are blocked until real GenImage samples are loaded.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

from src.forensic.bitplanes import extract_bit_planes, compose_low_bit_planes
from src.forensic.normalization import normalize_noise_thresholding, normalize_noise_scaling
from src.forensic.mgps import (
    get_directional_gradient_kernels,
    compute_patch_divergence_scores,
    maximum_gradient_patch_selection
)
from src.data.dataset import GenImageDataset

print("All LOTA forensic modules successfully loaded.")

## 1. Dataset Readiness & Sample Loading Gate

In [ ]:
DATASET_ROOT = "../data/GenImage"

def check_genimage_readiness(root_dir=DATASET_ROOT):
    if not os.path.exists(root_dir):
        return False, "Dataset root directory not found."
    ds = GenImageDataset(root_dir=root_dir, split="val", use_mock_data=False)
    if ds.use_mock_data or len(ds.samples) < 2:
        return False, "Insufficient real/fake image files in dataset directory."
    return True, f"GenImage dataset READY with {len(ds.samples)} samples across {len(ds.generators)} generators."

is_ready, status_msg = check_genimage_readiness()

if is_ready:
    print(f"[STATUS]: REAL-DATA VALIDATION READY!\n{status_msg}")
    MODE = "REAL_GENIMAGE"
    OUTPUT_FILENAME = "../experiments/visualizations/forensic_decomposition_genimage_real_vs_fake.png"
else:
    print("\n" + "*" * 80)
    print("  [WARNING] GenImage dataset was not found.")
    print("  This notebook is currently running in SYNTHETIC MATHEMATICAL VALIDATION MODE.")
    print("  The generated visualization validates mathematical pipeline mechanics only.")
    print("  It MUST NOT be interpreted as evidence of real-image or AI-generated-image forensic behavior.")
    print("  Qualitative comparisons with Figure 3 of the LOTA paper are DISABLED until real GenImage samples are available.")
    print("*" * 80 + "\n")
    MODE = "SYNTHETIC"
    OUTPUT_FILENAME = "../experiments/visualizations/forensic_decomposition_synthetic.png"

## 2. Load Sample Image Pair

In [ ]:
def get_sample_pair(mode=MODE, image_size=256):
    if mode == "REAL_GENIMAGE":
        ds = GenImageDataset(root_dir=DATASET_ROOT, split="val", use_mock_data=False)
        real_item = next(s for s in ds if s["label"] == 0)
        fake_item = next(s for s in ds if s["label"] == 1)
        real_np = real_item["raw_image"].numpy().transpose(1, 2, 0).astype(np.uint8)
        fake_np = fake_item["raw_image"].numpy().transpose(1, 2, 0).astype(np.uint8)
        labels = (f"Real Sample ({real_item['generator']})", f"AI Sample ({fake_item['generator']})")
    else:
        # Controlled synthetic sample pair for mathematical verification
        np.random.seed(42)
        x, y = np.meshgrid(np.linspace(0, 255, image_size), np.linspace(0, 255, image_size))
        base = (0.5 * x + 0.5 * y).astype(np.uint8)
        real_np = np.stack([base, base, base], axis=-1)
        
        fake_np = real_np.copy()
        artifact = np.random.randint(0, 8, (image_size, image_size, 3), dtype=np.uint8)
        fake_np = (fake_np & np.uint8(248)) | artifact
        labels = ("Synthetic Flat Sample [Mathematical Test]", "Synthetic Low-Bit Modulated Sample [Mathematical Test]")
    return real_np, fake_np, labels

sample_a, sample_b, (label_a, label_b) = get_sample_pair()
print(f"Sample A: {label_a} | Shape: {sample_a.shape}")
print(f"Sample B: {label_b} | Shape: {sample_b.shape}")

## 3. Execute 6-Stage Forensic Pipeline
1. **Input RGB Tensor** ($256 \times 256$)
2. **Bit-Plane Slicing** ($k=0\dots7$)
3. **Low-Bit Composition** ($z^c = \sum_{k=0}^2 2^k x_k^c$)
4. **Zero-Centering Threshold Normalization** ($\tilde{z}$)
5. **MGPS 4-Directional Gradient Convolutions & Divergence Map** ($g_p$)
6. **Argmax Patch Selection** ($32 \times 32$ Patch Overlay)

In [ ]:
def analyze_forensic_pipeline(img_np):
    img_tensor = torch.from_numpy(img_np).permute(2, 0, 1).unsqueeze(0).float()
    planes = extract_bit_planes(img_np)
    z_composed = compose_low_bit_planes(img_tensor, bit_indices=[0, 1, 2])
    z_norm = normalize_noise_thresholding(z_composed)
    patch, idx = maximum_gradient_patch_selection(z_norm, patch_size=32, strategy="max_gradient")
    
    num_h, num_w = 256 // 32, 256 // 32
    unfolded = z_norm.view(1, 3, num_h, 32, num_w, 32).permute(0, 2, 4, 1, 3, 5).contiguous().view(1, num_h*num_w, 3, 32, 32)
    divergence_scores = compute_patch_divergence_scores(unfolded).view(num_h, num_w).cpu().numpy()
    
    return planes, z_composed, z_norm, divergence_scores, patch, idx.item()

planes_a, z_a, znorm_a, div_a, patch_a, idx_a = analyze_forensic_pipeline(sample_a)
planes_b, z_b, znorm_b, div_b, patch_b, idx_b = analyze_forensic_pipeline(sample_b)

## 4. Visual Comparison & Provenance-Aware Artifact Generation

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9), dpi=150)

# Row 1: Sample A Pipeline
axes[0, 0].imshow(sample_a)
axes[0, 0].set_title(f"{label_a}: Input (256x256)")
axes[0, 0].axis("off")

axes[0, 1].imshow(znorm_a[0].permute(1, 2, 0).cpu().numpy() / 255.0)
axes[0, 1].set_title(f"{label_a}: Thresholded Noise ($\\tilde{z}$)")
axes[0, 1].axis("off")

im0 = axes[0, 2].imshow(div_a, cmap="hot")
plt.colorbar(im0, ax=axes[0, 2], fraction=0.046)
axes[0, 2].set_title(f"MGPS Divergence ($g_p$) [Best #{idx_a}]")
axes[0, 2].axis("off")

axes[0, 3].imshow(patch_a[0].permute(1, 2, 0).cpu().numpy() / 255.0)
axes[0, 3].set_title(f"Selected 32x32 Patch")
axes[0, 3].axis("off")

# Row 2: Sample B Pipeline
axes[1, 0].imshow(sample_b)
axes[1, 0].set_title(f"{label_b}: Input (256x256)")
axes[1, 0].axis("off")

axes[1, 1].imshow(znorm_b[0].permute(1, 2, 0).cpu().numpy() / 255.0)
axes[1, 1].set_title(f"{label_b}: Thresholded Noise ($\\tilde{z}$)")
axes[1, 1].axis("off")

im1 = axes[1, 2].imshow(div_b, cmap="hot")
plt.colorbar(im1, ax=axes[1, 2], fraction=0.046)
axes[1, 2].set_title(f"MGPS Divergence ($g_p$) [Best #{idx_b}]")
axes[1, 2].axis("off")

axes[1, 3].imshow(patch_b[0].permute(1, 2, 0).cpu().numpy() / 255.0)
axes[1, 3].set_title(f"Selected 32x32 Patch")
axes[1, 3].axis("off")

os.makedirs(os.path.dirname(OUTPUT_FILENAME), exist_ok=True)
plt.tight_layout()
plt.savefig(OUTPUT_FILENAME, dpi=200, bbox_inches="tight")
plt.show()
print(f"[SUCCESS] Visual artifact saved to: {OUTPUT_FILENAME}")

## 5. Qualitative Findings & Paper Figure 3 Comparison

```
QUALITATIVE COMPARISON POLICY:
- In SYNTHETIC mode: Qualitative observations regarding AI vs Natural image distributions are BLOCKED.
- In REAL GENIMAGE mode: Compare observations with Figure 3 of the paper without forcing identical matches.
```

### Documented Observations:
- When evaluating real GenImage samples, observe whether the lowest bit planes ($k=0, 1, 2$) in AI-generated images show higher high-frequency spatial variation and directional gradient scores ($g_p$) than natural images, as hypothesized in the LOTA paper.
- Any discrepancies should be investigated as research questions (e.g. compression artifacts, generator type differences, resolution effects) rather than hidden.